In [4]:
import pandas as pd

movies = pd.read_csv(
    r"C:\surya\ml\Movie-Recommendation-System\data\raw\movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["movieId", "title", "genres"]
)

ratings = pd.read_csv(
    r"C:\surya\ml\Movie-Recommendation-System\data\raw\ratings.dat",
    sep="::",
    engine="python",
    names=["userId", "movieId", "rating", "timestamp"]
)

users = pd.read_csv(
    r"C:\surya\ml\Movie-Recommendation-System\data\raw\users.dat",
    sep="::",
    engine="python",
    names=["userId", "gender", "age", "occupation", "zipCode"]
)

In [6]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [8]:
movies.shape
ratings.shape
users.shape

(6040, 5)

In [9]:
movies.head()
ratings.head()
users.head()

,userId,gender,age,occupation,zipCode
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [10]:
print("Movies Shape:", movies.shape)
print("Ratings Shape:", ratings.shape)
print("Users Shape:", users.shape)

Movies Shape: (3883, 3)
Ratings Shape: (1000209, 4)
Users Shape: (6040, 5)


In [11]:
print("Movies Info:", movies.info())
print("Ratings Info:", ratings.info())
print("Users Info:", users.info())


<class 'pandas.DataFrame'>
RangeIndex: 3883 entries, 0 to 3882
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  3883 non-null   int64
 1   title    3883 non-null   str  
 2   genres   3883 non-null   str  
dtypes: int64(1), str(2)
memory usage: 91.1 KB
Movies Info: None
<class 'pandas.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   userId     1000209 non-null  int64
 1   movieId    1000209 non-null  int64
 2   rating     1000209 non-null  int64
 3   timestamp  1000209 non-null  int64
dtypes: int64(4)
memory usage: 30.5 MB
Ratings Info: None
<class 'pandas.DataFrame'>
RangeIndex: 6040 entries, 0 to 6039
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   userId      6040 non-null   int64
 1   gender      6040 non-null   str  
 2   age  

In [12]:
print("Movies Null Values:", movies.isnull().sum())
print("Ratings Null Values:", ratings.isnull().sum())
print("Users Null Values:", users.isnull().sum())

Movies Null Values: movieId    0
title      0
genres     0
dtype: int64
Ratings Null Values: userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
Users Null Values: userId        0
gender        0
age           0
occupation    0
zipCode       0
dtype: int64


In [13]:
ratings["rating"].describe()

count    1.000209e+06
mean     3.581564e+00
std      1.117102e+00
min      1.000000e+00
25%      3.000000e+00
50%      4.000000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64

In [14]:
ratings["rating"].value_counts().sort_index()

rating
1     56174
2    107557
3    261197
4    348971
5    226310
Name: count, dtype: int64

In [17]:
ratings.groupby("movieId")["rating"].mean().sort_values(ascending=False).head(10)

movieId
3280    5.0
3382    5.0
3172    5.0
3233    5.0
787     5.0
1830    5.0
989     5.0
3656    5.0
3607    5.0
3881    5.0
Name: rating, dtype: float64

In [18]:
print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Users:", users.shape)

print("\nAverage Rating:")
print(ratings["rating"].mean())

print("\nUnique Users:")
print(ratings["userId"].nunique())

print("\nUnique Movies Rated:")
print(ratings["movieId"].nunique())

print("\nRating Distribution:")
print(ratings["rating"].value_counts().sort_index())

Movies: (3883, 3)
Ratings: (1000209, 4)
Users: (6040, 5)

Average Rating:
3.581564453029317

Unique Users:
6040

Unique Movies Rated:
3706

Rating Distribution:
rating
1     56174
2    107557
3    261197
4    348971
5    226310
Name: count, dtype: int64


In [19]:
movie_stats = ratings.groupby("movieId").agg(
    rating_count=("rating", "count"),
    avg_rating=("rating", "mean")
)

movie_stats = movie_stats.reset_index()

movie_stats = movie_stats.merge(
    movies,
    on="movieId"
)

movie_stats.head()

,movieId,rating_count,avg_rating,title,genres
0,1,2077,4.146846,Toy Story (1995),Animation|Children's|Comedy
1,2,701,3.201141,Jumanji (1995),Adventure|Children's|Fantasy
2,3,478,3.016736,Grumpier Old Men (1995),Comedy|Romance
3,4,170,2.729412,Waiting to Exhale (1995),Comedy|Drama
4,5,296,3.006757,Father of the Bride Part II (1995),Comedy


In [20]:
top_movies = movie_stats.sort_values(
    by="rating_count",
    ascending=False
)

top_movies[
    ["title", "rating_count", "avg_rating"]
].head(10)

,title,rating_count,avg_rating
2651,American Beauty (1999),3428,4.317386
253,Star Wars: Episode IV - A New Hope (1977),2991,4.453694
1106,Star Wars: Episode V - The Empire Strikes Back...,2990,4.292977
1120,Star Wars: Episode VI - Return of the Jedi (1983),2883,4.022893
466,Jurassic Park (1993),2672,3.763847
1848,Saving Private Ryan (1998),2653,4.337354
575,Terminator 2: Judgment Day (1991),2649,4.058513
2374,"Matrix, The (1999)",2590,4.315830
1178,Back to the Future (1985),2583,3.990321
579,"Silence of the Lambs, The (1991)",2578,4.351823


In [21]:
def recommend_popular_movies(n=20):
    return top_movies[
        ["title", "rating_count", "avg_rating"]
    ].head(n)

recommend_popular_movies()

,title,rating_count,avg_rating
2651,American Beauty (1999),3428,4.317386
253,Star Wars: Episode IV - A New Hope (1977),2991,4.453694
1106,Star Wars: Episode V - The Empire Strikes Back...,2990,4.292977
1120,Star Wars: Episode VI - Return of the Jedi (1983),2883,4.022893
466,Jurassic Park (1993),2672,3.763847
1848,Saving Private Ryan (1998),2653,4.337354
575,Terminator 2: Judgment Day (1991),2649,4.058513
2374,"Matrix, The (1999)",2590,4.315830
1178,Back to the Future (1985),2583,3.990321
579,"Silence of the Lambs, The (1991)",2578,4.351823


In [22]:
movie_stats = ratings.groupby("movieId").agg(
    rating_count=("rating", "count"),
    avg_rating=("rating", "mean")
)

movie_stats = movie_stats.reset_index()

movie_stats = movie_stats.merge(
    movies,
    on="movieId"
)

top_movies = movie_stats.sort_values(
    by="rating_count",
    ascending=False
)

top_movies[
    ["title", "rating_count", "avg_rating"]
].head(20)

,title,rating_count,avg_rating
2651,American Beauty (1999),3428,4.317386
253,Star Wars: Episode IV - A New Hope (1977),2991,4.453694
1106,Star Wars: Episode V - The Empire Strikes Back...,2990,4.292977
1120,Star Wars: Episode VI - Return of the Jedi (1983),2883,4.022893
466,Jurassic Park (1993),2672,3.763847
1848,Saving Private Ryan (1998),2653,4.337354
575,Terminator 2: Judgment Day (1991),2649,4.058513
2374,"Matrix, The (1999)",2590,4.315830
1178,Back to the Future (1985),2583,3.990321
579,"Silence of the Lambs, The (1991)",2578,4.351823


In [23]:
movie_stats = ratings.groupby("movieId").agg(
    rating_count=("rating", "count"),
    avg_rating=("rating", "mean")
)

movie_stats = movie_stats.reset_index()

movie_stats = movie_stats.merge(
    movies,
    on="movieId"
)

top_movies = movie_stats.sort_values(
    by="rating_count",
    ascending=False
)

top_movies[
    ["title", "rating_count", "avg_rating"]
].head(20)

,title,rating_count,avg_rating
2651,American Beauty (1999),3428,4.317386
253,Star Wars: Episode IV - A New Hope (1977),2991,4.453694
1106,Star Wars: Episode V - The Empire Strikes Back...,2990,4.292977
1120,Star Wars: Episode VI - Return of the Jedi (1983),2883,4.022893
466,Jurassic Park (1993),2672,3.763847
1848,Saving Private Ryan (1998),2653,4.337354
575,Terminator 2: Judgment Day (1991),2649,4.058513
2374,"Matrix, The (1999)",2590,4.315830
1178,Back to the Future (1985),2583,3.990321
579,"Silence of the Lambs, The (1991)",2578,4.351823


In [24]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(movies["genres"])

print(tfidf_matrix.shape)

(3883, 20)


In [41]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix)

print(cosine_sim.shape)

(3883, 3883)


In [27]:
indices = pd.Series(
    movies.index,
    index=movies["title"]
).drop_duplicates()

In [40]:
def get_recommendations(title, n=10):

    idx = indices[title]

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:n+1]

    movie_indices = [
        i[0]
        for i in similarity_scores
    ]

    return movies["title"].iloc[movie_indices]

In [32]:
print("Recommendations for 'Toy Story (1995)':", get_recommendations("Toy Story (1995)"))
print("Recommendations for 'Jumanji (1995)':", get_recommendations("Jumanji (1995)"))
print("Recommendations for 'Titanic (1997)':", get_recommendations("Titanic (1997)"))

Recommendations for 'Toy Story (1995)': 1050            Aladdin and the King of Thieves (1996)
2072                          American Tail, An (1986)
2073        American Tail: Fievel Goes West, An (1991)
2285                         Rugrats Movie, The (1998)
2286                              Bug's Life, A (1998)
3045                                Toy Story 2 (1999)
3542                             Saludos Amigos (1943)
3682                                Chicken Run (2000)
3685    Adventures of Rocky and Bullwinkle, The (2000)
12                                        Balto (1995)
Name: title, dtype: str
Recommendations for 'Jumanji (1995)': 55                         Kids of the Round Table (1995)
59                     Indian in the Cupboard, The (1995)
124                     NeverEnding Story III, The (1994)
996                       Escape to Witch Mountain (1975)
1898                                     Labyrinth (1986)
1936                                  Goonies, The (1985)


In [33]:
user_movie_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

In [34]:
user_movie_matrix = user_movie_matrix.fillna(0)

In [35]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_movie_matrix)

In [36]:
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

In [37]:
user_similarity_df.loc[1].sort_values(ascending=False).head(10)

userId
1       1.000000
5343    0.412117
5190    0.411899
1481    0.392110
1283    0.386597
5705    0.360898
5762    0.355600
1359    0.350329
1476    0.339496
541     0.334864
Name: 1, dtype: float64

In [39]:
! pip install streamlit


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
